# Experiment 3: Answer Quality

Inspect 79 paired questions sampled with a cap of 25 per retrieval subgroup. Each arm uses its own frozen five passages. IMEI is already one of those 79 questions.

**Live execution is the default.** `MODE="live"` runs this notebook’s model calls with the configured OpenAI-compatible client. Set `OPENROUTER_API_KEY` before starting the kernel. Prepared document embeddings and frozen experimental inputs remain fixed; passage embeddings are not rebuilt.


**Reference input scope:** All four experiments select from the frozen **15,585 fully covered question IDs**. Targeted reranking uses **500 opportunities + 200 controls**; answer quality samples **up to 25 per resulting subgroup**, and latency samples from those answers. Passage embeddings are reused; live question embedding, reranking, generation and judging calls produce outputs in this notebook.  

**Hosted generator limitation:** Both arms use the same generator model ID and requested settings. OpenRouter may route requests to different serving providers, introducing backend variation alongside the context changes. This run does not isolate that variation; the separate three-prompt extension pins its serving provider for its own controlled comparison.

**Workflow:** Experiment 1 (Baseline) --> Diagnosis --> Experiment 2 (Targeted Reranking) --> **this Experiment 3 (Answer Quality)** --> Experiment 4 (Latency Tradeoff)


**Recorded reference run:** 10 November 2026. Saved displays belong to that execution. A new Live run computes its own results; use its current tables rather than treating reference findings as predictions.

## 1. Set up the answer-quality study

**What:** use the frozen sample of up to 25 questions per resulting subgroup and execute paired Qwen generation on each arm’s exact five passages. Inspect the answers, requests, responses and scores.

**Why:** measure new answer quality on the declared frozen context selections. Use Python 3.14.6 and the pinned project dependencies; Live execution includes generation and RAGAS judging.


In [1]:
STUDY = 'answers'
from pathlib import Path
import os, sys
from IPython.display import display, Markdown, HTML
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "configs/tutorial_inputs.json").is_file())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from shared_scoring.live_tutorial import TutorialRun
from shared_scoring.provider import embedding_payload, rerank_payload
from shared_scoring.tutorial_views import TutorialViews
from shared_scoring.notebook_display import display_table, details_html
from shared_scoring.io import read_json
import inspect, json
import pandas as pd

MODE = os.environ.get("NOTEBOOK_MODE", "live").lower()  # live | replay | plan
RUN_ID = os.environ.get("TUTORIAL_RUN_ID") or None
RESUME = os.environ.get("TUTORIAL_RESUME") == "1"  # explicit same-run resume only

tutorial = TutorialRun(PROJECT, STUDY, mode=MODE, run_id=RUN_ID, resume=RESUME)
tutorial.preflight(create_client=False)  # assets, credentials and offline tests; no model requests
client = None
if MODE == "live":
    from openai import OpenAI
    import httpx
    # Standard OpenAI-style client; the transport records this run and enforces caps.
    client = OpenAI(
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url=tutorial.retrieval["api_base"],
        max_retries=0,
        timeout=tutorial.retrieval["timeout_seconds"],
        http_client=httpx.Client(transport=tutorial.transport()),
    )
    tutorial.bind_client(client)
# Selecting Live authorizes this notebook's declared workflow; no second paid-call switch.

view = TutorialViews(tutorial)
view.show("configuration")


Setting,Value
Embedding model,qwen/qwen3-embedding-4b
Embedding dimensions,2560
Reranker model,qwen/qwen3-reranker-8b
Generator model,qwen/qwen3-30b-a3b-instruct-2507
Candidate / context budget,30 / 5 passages
Eligible questions in this experiment,79
Generation limit,400 tokens


## 2. Keep the frozen sample and unequal subgroup sizes

Sample up to 25 questions per planned retrieval subgroup, using seed 42 after reranking and before answer scoring. The parent study contains 500 opportunities and 200 controls.

The frozen sample has **25 opportunity gains, 25 unchanged cases, 4 losses, and 25 all-support controls: 79 questions**. The planned control subgroup with support absent from the top thirty has **no eligible cases in this sampled parent cohort**. It is not a pending execution stage.

| Group | Qualification |
|---|---|
| Opportunity gain | Reranked Recall@5 is higher. |
| Opportunity unchanged | Recall@5 is unchanged. |
| Opportunity loss | Reranked Recall@5 is lower. |
| Control: all support already present | Embedding Recall@5 = Recall@30 = 1. |
| Control: some support absent from top 30 | Embedding Recall@5 = Recall@30 < 1; no sampled cases here. |

The group names describe retrieval, not answer quality. Each question keeps the same thirty candidate IDs and texts across arms; ranking and the selected top five can differ. Selection is not proportional to parent-group sizes, so report groups separately rather than treating their pooled average as population-wide improvement. The four losses support exploratory case inspection.

**IMEI (1053665) is already a sampled opportunity-gain question.** Its walkthrough display does not add an observation.


Subgroup sizes are unequal because sampling is capped at 25 per group: only four opportunity-loss questions and no eligible missing-support controls exist in the frozen parent cohort.

These assignments are fixed by the public input recipe. A new targeted-reranking run can have different outcomes; it does not silently resample this answer-quality study.

In [2]:
view.show('questions')
view.show('input_contract')


subgroup
control_all_support_already_present    25
opportunity_gain                       25
opportunity_loss                        4
opportunity_unchanged                  25
Name: questions, dtype: int64

query_id,question,subgroup,diagnostic
1031584,what is the technology behind spoofing apps,control_all_support_already_present,False
query_id,question,subgroup,diagnostic
1034319,what is the population of north ko,control_all_support_already_present,False
1036388,cross reactive allergens,control_all_support_already_present,False
1036720,what is the meaning ha in mathematics,control_all_support_already_present,False
1044161,what is the basic principle of systema,control_all_support_already_present,False
1045113,what is the active ingredient in ventolin hfa,control_all_support_already_present,False
1046747,what is rx provigil u,control_all_support_already_present,False
1054187,corn is used for what foods,control_all_support_already_present,False
1055717,what is epic database for healthcare,control_all_support_already_present,False


{'questions': 79, 'candidate_rows': 2370, 'candidates_per_question': 30}

## 3. Hold the generation contract fixed

**What:** both arms used the same generator, system prompt, temperature, seed and 400-token output limit. Submission order alternated by sample position.

**Why:** compare the effect of each arm’s five selected passages on the answer.

Embedding RAG used embedding ranks 1–5. Embedding + reranking RAG used reranker ranks 1–5 from the same 30 candidates. Passage text and order are frozen, with the inherited 1,800-character passage budget. No additional context truncation was introduced.

All original evaluation references are retained in their recorded order. References and support labels were held out of generator prompts. Expand the settings to inspect the exact evidence-only prompt and input contract.

In [3]:
view.show('prompts')
def generate_api(messages, settings, routing=None):
    # The actual SDK call; the tutorial invokes it for each declared question/arm.
    return client.chat.completions.create(
        model=settings["model"], messages=messages,
        temperature=settings["temperature"], max_tokens=settings["max_tokens"],
        **settings["extras"], **({"extra_body": {"provider": routing}} if routing else {}))

generation_run = tutorial.generate(generate_api)
if generation_run is not None:
    display(generation_run)
view.show('completion')


**baseline / original**

**reranked / original**

generate: saved for CURRENT LIVE RUN 20261111T034102Z-e3b73fea18b4


{'answers': 158, 'questions': 79, 'run_id': '20261111T034102Z-e3b73fea18b4'}

Measure,Value
run_id,20261111T034102Z-e3b73fea18b4
questions,79
successful_answers,158
complete_pairs,79
main_complete_pairs,79
truncated_answers,0
request_outcomes,{'ok': 158}


## 4. Check completion, failures and truncation

Use this selected run’s completion and request tables to inspect complete pairs, successful answers, failed attempts and truncation. The frozen design plans 79 pairs; actual completion must be checked rather than assumed.

Displaying IMEI shows one pair from the sample; it does not generate another sampled pair or alter the denominators.

In [4]:
view.show('completion')
view.show('attempts')


Measure,Value
run_id,20261111T034102Z-e3b73fea18b4
questions,79
successful_answers,158
complete_pairs,79
main_complete_pairs,79
truncated_answers,0
request_outcomes,{'ok': 158}


stage,query_id,arm,attempt,status,http_status,elapsed_seconds,backoff_seconds
generation,1084423,baseline,1,ok,200,3.1682,0.0000
stage,query_id,arm,attempt,status,http_status,elapsed_seconds,backoff_seconds
generation,1086439,reranked,1,ok,200,1.7212,0.0000
generation,330883,baseline,1,ok,200,5.3753,0.0000
generation,1100897,baseline,1,ok,200,3.3643,0.0000
generation,1084140,reranked,1,ok,200,3.2443,0.0000
generation,1087589,reranked,1,ok,200,2.7604,0.0000
generation,1088380,reranked,1,ok,200,2.4022,0.0000
generation,1085285,reranked,1,ok,200,0.9931,0.0000
generation,1084132,baseline,1,ok,200,3.2844,0.0000


## 5. Explain the answer metrics

**What:** use reference overlap for deterministic scoring and model judgment for support and relevance.

**Why:** these metrics answer different questions about answer quality.

Our preprocessing removes passage-ID citations while retaining answer text. Hugging Face SQuAD then normalizes case, punctuation, articles and whitespace and uses its best-reference scoring across all retained references. **Normalized exact match and Token F1 are percentages (0–100).** The existing custom scores use **0–1**.

RAGAS **Faithfulness** checks answer claims against that arm’s exact five passages. **Answer relevancy** generates questions from the generated answer and compares them with the original question using the configured evaluation embeddings. `embedding_question_policy="evaluate"` permits this evaluation embedding action during live scoring.

Judge metrics are estimates, separate from human correctness. Each metric compares only its own matching successful pairs; failed or missing evaluations are reported separately.

In [5]:
from shared_scoring.answers import score_squad_answers
from shared_scoring.judge import run_ragas_evaluation
display(Markdown("```python\n" + inspect.getsource(score_squad_answers) + "\n```"))


```python
def score_squad_answers(inputs, metric, citation_pattern, legacy_citation_pattern):
    """Score each saved answer, keeping truncation and the old scorer comparison."""
    records = []
    for row in inputs:
        result = score_squad_answer(metric, row['answer'], row['reference_answers'], row['query_id'], citation_pattern)
        custom = score_custom_answer(row['answer'], row['reference_answers'], legacy_citation_pattern)
        for key, saved in [('token_f1', 'existing_token_f1_0_1'), ('normalized_em', 'existing_normalized_em_0_1')]:
            if saved in row and abs(custom[key] - row[saved]) > 1e-12:
                raise ValueError(f"Existing custom score changed: {row['query_id']} / {row['arm']} / {key}")
        records.append({key: row[key] for key in ['query_id', 'question', 'arm', 'path', 'answer', 'finish_reason', 'truncated',
                                                 'annotated_support_recall_at_5', 'existing_token_f1_0_1', 'existing_normalized_em_0_1']} |
                       {'squad_answer': remove_passage_citations(row['answer'], citation_pattern),
                        'reference_answers': json.dumps(row['reference_answers'], ensure_ascii=False),
                        'squad_exact_match_pct': result['exact_match'], 'squad_token_f1_pct': result['f1'],
                        'token_f1_delta_0_1': result['f1'] / 100 - custom['token_f1'],
                        'normalized_em_delta_0_1': result['exact_match'] / 100 - custom['normalized_em']})
    return pd.DataFrame(records)

```

## 6. Run answer scoring

**What:** compute local SQuAD scores and hosted RAGAS judgments for this run’s answers.

**Why:** evaluate reference overlap, evidence support and question relevance separately.

Hugging Face `evaluate.load` and `metric.compute` provide multiple-reference SQuAD scoring. The pinned official SQuAD source is available locally for reproducibility.

RAGAS uses `llm_factory`, `Faithfulness` and `AnswerRelevancy`. Live judging requires `OPENROUTER_API_KEY` in the environment. Helpers handle prompts, pacing, capped retries, timing and evaluation records. The framework source is inspectable in the appendix.

In [6]:
from shared_scoring.notebook_display import collapsible_output

with collapsible_output("Scoring details — expand/collapse"):
    score_run = await tutorial.score()
    if score_run is not None:
        display(score_run)
    view.show('judge_outcomes')


## 7. Compare paired answer quality

**Recorded reference run:** numerical observations below describe the saved execution. New Live results may differ; the computed tables describe the selected run.

This **recorded reference run** generated all **79 sampled pairs** and computed deterministic scores. RAGAS outcome tables report successful, failed and missing evaluations separately; paired judge summaries use only matching successful pairs. Human correctness review remains pending.

Token F1 differences (reranked minus embedding) are:

- opportunity gain: **+3.79 percentage points** (25 pairs; 95% bootstrap interval **-0.09 to +7.95**)
- opportunity unchanged: **-0.62 percentage points** (25 pairs; 95% bootstrap interval **-3.78 to +2.35**)
- opportunity loss: **-8.72 percentage points** (4 pairs; 95% bootstrap interval **-10.20 to -7.37**)
- control all support already present: **+2.24 percentage points** (25 pairs; 95% bootstrap interval **-1.56 to +6.86**)

Failed or missing judge outcomes: **0 faithfulness**, **0 answer relevancy**. Use each metric’s matched-pair counts and uncertainty. Retrieval improvement does not establish answer improvement for every question.

In [7]:
view.show('scores')


query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1031584,what is the technology behind spoofing apps,baseline,baseline,"The technology behind spoofing apps primarily involves VoIP (Voice-over-Internet-Protocol) technology, which enables voice communications over the internet and facilitates caller ID spoofing [p_d0692237237e214e45f6]. Additionally, less common methods include emulating the Bell 202 FSK signal (orange boxing) and accessing the SS7 network to make calls appear as if they originate from another phone number [p_cff38286a9d88db1a72f]. These technologies allow spoofing apps to manipulate caller ID information, though their use can be legally restricted depending on intent [p_7f89a766dddc564217a2].",stop,False,1.0000,0.1579,0.0000,"The technology behind spoofing apps primarily involves VoIP (Voice-over-Internet-Protocol) technology, which enables voice communications over the internet and facilitates caller ID spoofing . Additionally, less common methods include emulating the Bell 202 FSK signal (orange boxing) and accessing the SS7 network to make calls appear as if they originate from another phone number . These technologies allow spoofing apps to manipulate caller ID information, though their use can be legally restricted depending on intent .","[""The technology behind spoofing apps is call spoofing technologies.""]",0.0000,15.7895,0.0000,0.0000,1.0000,ok,0.8097,ok,control_all_support_already_present,False,75.0000,Nebius,20261111T034102Z-e3b73fea18b4
query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1031584,what is the technology behind spoofing apps,reranked,reranked,"The technology behind spoofing apps primarily involves VoIP (Voice-over-Internet-Protocol) technology, which enables voice communications over the internet rather than traditional phone lines, allowing for easy manipulation of Caller ID information [p_d0692237237e214e45f6]. Additionally, spoofing can be achieved through emulation of Bell 202 FSK signals (orange boxing) or access to the SS7 network, both of which can make calls appear to originate from another number, even to telephone company operators [p_cff38286a9d88db1a72f]. In the context of app fraud, SDK spoofing or replay attacks involve mimicking communication from a legitimate app installation to deceive tracking systems [p_949db4d42be13281e4ad].",stop,False,1.0000,0.1075,0.0000,"The technology behind spoofing apps primarily involves VoIP (Voice-over-Internet-Protocol) technology, which enables voice communications over the internet rather than traditional phone lines, allowing for easy manipulation of Caller ID information . Additionally, spoofing can be achieved through emulation of Bell 202 FSK signals (orange boxing) or access to the SS7 network, both of which can make calls appear to originate from another number, even to telephone company operators . In the context of app fraud, SDK spoofing or replay attacks involve mimicking communication from a legitimate app installation to deceive tracking systems .","[""The technology behind spoofing apps is call spoofing technologies.""]",0.0000,10.7527,-0.0000,0.0000,0.9231,ok,0.8406,ok,control_all_support_already_present,False,93.0000,Nebius,20261111T034102Z-e3b73fea18b4
1034319,what is the population of north ko,reranked,reranked,"The population of North Korea was 25,368,620 as of the end of 2016 [p_c

subgroup,comparison,metric,matched_pairs,original_mean,variant_mean,mean_delta,bootstrap_low,bootstrap_high
control_all_support_already_present,reranked,squad_token_f1_pct,25,44.1642,46.4051,2.2410,-1.5618,6.8635
subgroup,comparison,metric,matched_pairs,original_mean,variant_mean,mean_delta,bootstrap_low,bootstrap_high
control_all_support_already_present,reranked,faithfulness,25,0.9914,0.9452,-0.0462,-0.0882,-0.0131
control_all_support_already_present,reranked,answer_relevancy,25,0.7905,0.8678,0.0773,-0.0076,0.1869
control_all_support_already_present,reranked,answer_words,25,50.8800,50.8000,-0.0800,-6.0420,5.5600
opportunity_gain,reranked,squad_token_f1_pct,25,25.4266,29.2203,3.7938,-0.0941,7.9472
opportunity_gain,reranked,faithfulness,25,0.9037,0.9488,0.0451,-0.0063,0.1097
opportunity_gain,reranked,answer_relevancy,25,0.7950,0.7687,-0.0263,-0.1003,0.0250
opportunity_gain,reranked,answer_words,25,53.2800,56.2800,3.0000,-7.3200,14.7610
opportunity_loss,reranked,squad_token_f1_pct,4,29.3242,20.5992,-8.7250,-10.2049,-7.3682


## 8. Worked example: IMEI answer quality

**Recorded reference run:** numerical observations below describe the saved execution. New Live results may differ; the computed tables describe the selected run.

Inspect **1053665** with its frozen **152,576-passage corpus** contexts and this **recorded reference run’s** new answers and scores. The frozen context selection has Recall@5 **0.5 → 1.0**. Token F1 changes **27.69 → 51.16%**; judged faithfulness changes **0.64 → 1.00**; answer relevancy changes **0.8897 → 0.8610**. Inspect claim support in the displayed passages; F1 measures reference overlap and judge scores remain estimates. Missing judge values are reported as missing, never as zero.

IMEI already belongs to the frozen **79-question sample**. Its diagnostic display does not add an observation or change subgroup membership. These frozen answer-quality contexts are explicit inputs; they are not replaced by the new Experiment 2 rankings.

In [8]:
from shared_scoring.notebook_display import collapsible_output

with collapsible_output('IMEI evidence, answers and scores — expand/collapse'):
    view.show('walkthrough')


query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1053665,what is imei and where do i find it,reranked,reranked,"The IMEI (International Mobile Equipment Identity) is a globally unique 15-digit number assigned to all cellular devices, which identifies the device within the mobile network [p_71838a0d93f9593a28b4]. You can find the IMEI on the original packaging of your device [p_ceb444d4a87718de91ee], under the battery on older phones [p_f446bafbb6b1eedb4db9], on the back of the device (e.g., iPhone, iPad, iPod touch) [p_fbac8ec80aec14e774ec], or in the device settings under General > This Device [p_16e01af39f1ab9cc5a66].",stop,False,1.0000,0.5116,0.0000,"The IMEI (International Mobile Equipment Identity) is a globally unique 15-digit number assigned to all cellular devices, which identifies the device within the mobile network . You can find the IMEI on the original packaging of your device , under the battery on older phones , on the back of the device (e.g., iPhone, iPad, iPod touch) , or in the device settings under General > This Device .","[""An International Mobile Equipment Identity number is a globally unique 15 digit number assigned to all cellular devices, which identifies your device within the mobile network, printed on a sticker that is affixed to the phone underneath the battery.""]",0.0000,51.1628,0.0000,0.0000,1.0000,ok,0.8610,ok,opportunity_gain,True,69.0000,Nebius,20261111T034102Z-e3b73fea18b4
query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1053665,what is imei and where do i find it,baseline,baseline,"The IMEI (International Mobile Equipment Identity) is a unique identifier assigned to mobile devices, used to identify valid devices and prevent theft. It is typically a 15-digit number, though CDMA networks use a 14-digit MEID, which is the first 14 digits of the IMEI. You can find the IMEI on the device's original packaging, on a sticker under the battery (for phones with removable batteries), on the back of the device (especially iPhones, iPads, and iPod touches), in the device settings (e.g., iPhone: Settings > General > About), or in the SIM card slot area (for older iPhone models). For Android devices, the IMEI can be found in the device settings or by dialing *#06# [p_f446bafbb6b1eedb4db9], [p_ceb444d4a87718de91ee], [p_16e01af39f1ab9cc5a66], [p_fbac8ec80aec14e774ec], [p_ba06e4d846eb6168282d].",stop,False,0.5000,0.2769,0.0000,"The IMEI (International Mobile Equipment Identity) is a unique identifier assigned to mobile devices, used to identify valid devices and prevent theft. It is typically a 15-digit number, though CDMA networks use a 14-digit MEID, which is the first 14 digits of the IMEI. You can find the IMEI on the device's original packaging, on a sticker under the battery (for phones with removable batteries), on the back of the device (especially iPhones, iPads, and iPod touches), in the device settings (e.g., iPhone: Settings > General > About), or in the SIM card slot area (for older iPhone models). For Android devices, the IMEI can be found in the device settings or by dialing *#06# , , , , .","[""An International Mobile Equipment Identity number is a globally unique 15 digit number assigned to all cellular devices, which identifies your device within the mobile network, printed on a sticker that is affixed to the phone underneath the battery.""]",0.0000

In [9]:
from shared_scoring.notebook_display import collapsible_output

with collapsible_output('IMEI prompts — expand/collapse'):
    view.show('prompts')


### Can generation use the evidence better?

**Recorded reference run:** numerical observations below describe the saved execution. New Live results may differ; the computed tables describe the selected run.

**Exploratory generation extension — recorded reference run incomplete:** 33 of 36 new answers are saved on the frozen 12-question subset under the pinned provider. A connection reset stopped query **1087577**, original prompt. Its one-attempt cap has been reached; one additional attempt awaits explicit approval. The other two unattempted variants remain pending. Extension judging has not run. Do not interpret the partial extension as completed results.

The **79-question main study is complete**, with all 158 answers and 316 judge outcomes saved. It remains separate from the extension.

In [10]:
from shared_scoring.notebook_display import collapsible_output

view.show('extension_contract')
# Generate the three prompt variants on the same frozen evidence.
with collapsible_output("Show remaining extension generation log — expand/collapse", preview_rows=1, preview_logs=True):
    extension_run = tutorial.generate(generate_api, extension=True)
    if extension_run is not None:
        display(extension_run)


{'control_all_support_already_present': ['1055717', '1088380', '1031584'],
 'opportunity_gain': ['1075969', '1087589', '1053665'],
 'opportunity_loss': ['1079518', '205107', '1064000'],
 'opportunity_unchanged': ['1084132', '1087577', '1035128']}

{'only': ['nebius/fp8'],
 'order': ['nebius/fp8'],
 'allow_fallbacks': False,
 'require_parameters': True}

**reranked / original**

**reranked / evidence_check**

**reranked / concise**

APIConnectionError: Connection error.

In [ ]:
from shared_scoring.notebook_display import collapsible_output

with collapsible_output("Show remaining extension scoring output — expand/collapse", preview_rows=1):
    extension_scores = await tutorial.score(extension=True)
    if extension_scores is not None:
        display(extension_scores)


In [ ]:
from shared_scoring.notebook_display import collapsible_output

with collapsible_output("Show remaining prompt results, examples and qualitative review — expand/collapse", preview_rows=1):
    view.show('extension')


## 9. Keep annotated retrieval scores separate

**What:** show the existing **Annotated-support Recall@5** on its **0–1** scale for this selected sample.

**Why:** explain the evidence changes behind the answer comparisons without changing the retrieval metric.

These are the inherited annotated-support scores for up to 25 questions per stratum, not full-cohort results or RAGAS Faithfulness. Support labels describe retrieval coverage; they do not establish answer correctness.

In [ ]:
view.show('retrieval_summary')


## 11. Diagnostic membership

IMEI **1053665** is included in the sampled opportunity-gain group and remains in all primary aggregate results. Effervescent **1100870** is absent from this 79-question sample. The Live-run membership check below displays both facts directly; repeated walkthroughs do not add or remove questions.

In [ ]:
view.show('diagnostic_membership')


## 12. Interpret the findings

Use the computed paired differences and uncertainty above.

Compare the paired changes within each subgroup, their confidence intervals, and failures; this section interprets the results without regenerating answers or rescoring them.

In [ ]:
view.show('findings')
tutorial.show(summary=False)


## Appendix: inspect the real framework entry points

**What:** read the actual helper implementations of Hugging Face `evaluate.load`/`metric.compute` and RAGAS `llm_factory`, `Faithfulness` and `AnswerRelevancy`.

**Why:** keep framework attribution verifiable while keeping setup and bulk execution outside the walkthrough. This cell displays source; it does not load a metric or construct a model client.

In [ ]:
display(Markdown("```python\n" + inspect.getsource(run_ragas_evaluation) + "\n```"))


## Appendix: verify the frozen inputs

Optional reproducibility check: verify question IDs, candidate ranks, evidence contexts and input integrity. The tutorial does not need checksum or notebook-revision details to explain the findings; those checks remain available in the helper and configuration files. No model calls are made.

In [ ]:
# Verify frozen inputs without printing checksums or notebook revision bookkeeping.
view.show('audit')
